In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc

# Parameter generation


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

# --- Define sampled parameters ---
gene_params_sampled = [
    "pi_on",            # k_on / (k_on + k_off)
    "k_on",             # directly sampled k_on
    "mrna_half_life",
    "protein_half_life",
    "k_prod_protein",
    "k_prod_mRNA"
]

interaction_params_scaled = [
    "n_gene_1_to_gene_2", "k_add_scaled_gene_1_to_gene_2",
    "n_gene_2_to_gene_1", "k_add_scaled_gene_2_to_gene_1",
    "n_gene_1_to_gene_3", "k_add_scaled_gene_1_to_gene_3",
    "n_gene_2_to_gene_3", "k_add_scaled_gene_2_to_gene_3"
]

param_names = (
    [f"{p}_gene_1" for p in gene_params_sampled] +
    [f"{p}_gene_2" for p in gene_params_sampled] +
    [f"{p}_gene_3" for p in gene_params_sampled] +
    interaction_params_scaled
)

# --- Bounds for sampled parameters ---
param_bounds = {
    # Gene-level
    "pi_on": (0.002, 0.4),                  # activation probability
    "k_on": (0.01, 3),                      # directly sampled
    "mrna_half_life": (0.6, 17),
    "protein_half_life": (7, 200),
    "k_prod_mRNA": (0.2, 60),
    "k_prod_protein": (19, 2700),
    # Interaction-level
    "n_gene_1_to_gene_2": (0.1, 5),
    "n_gene_2_to_gene_1": (0.1, 5),
    "n_gene_1_to_gene_3": (0.1, 5),
    "n_gene_2_to_gene_3": (0.1, 5),
    "k_add_scaled_gene_1_to_gene_2": (0.5, 10),
    "k_add_scaled_gene_2_to_gene_1": (0.5, 10),
    "k_add_scaled_gene_1_to_gene_3": (0.5, 10),
    "k_add_scaled_gene_2_to_gene_3": (0.5, 10),
}

bounds = (
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in interaction_params_scaled]
)

# --- Sampling configuration ---
n_valid_required = 25000
seed = 42

# Latin Hypercube Sampling (log10 for all except pi_on)
log_bounds_lower = [
    np.log10(b[0]) if "pi_on" not in param_names[i] else b[0]
    for i, b in enumerate(bounds)
]
log_bounds_upper = [
    np.log10(b[1]) if "pi_on" not in param_names[i] else b[1]
    for i, b in enumerate(bounds)
]

sampler = qmc.LatinHypercube(d=len(bounds), seed=seed)
sample = sampler.random(n=n_valid_required)

scaled_sample = np.empty_like(sample)
for i, name in enumerate(param_names):
    # Determine log bounds for every parameter (including pi_on)
    lower = np.log10(param_bounds["pi_on"][0]) if "pi_on" in name else log_bounds_lower[i]
    upper = np.log10(param_bounds["pi_on"][1]) if "pi_on" in name else log_bounds_upper[i]

    # Scale in log-space and exponentiate back
    scaled_log = qmc.scale(sample[:, [i]], [lower], [upper]).ravel()
    scaled_sample[:, i] = 10 ** scaled_log



df_sampled = pd.DataFrame(scaled_sample, columns=param_names)

# --- Convert to actual k_off and k_add ---
def convert_params(row):
    converted = {}

    for g in [1, 2, 3]:
        pi = row[f"pi_on_gene_{g}"]
        k_on = row[f"k_on_gene_{g}"]

        # Derive k_off from pi_on and k_on
        k_off = k_on * (1 - pi) / pi

        converted[f"k_on_gene_{g}"] = k_on
        converted[f"k_off_gene_{g}"] = k_off
        converted[f"mrna_half_life_gene_{g}"] = row[f"mrna_half_life_gene_{g}"]
        converted[f"protein_half_life_gene_{g}"] = row[f"protein_half_life_gene_{g}"]
        converted[f"k_prod_mRNA_gene_{g}"] = row[f"k_prod_mRNA_gene_{g}"]
        converted[f"k_prod_protein_gene_{g}"] = row[f"k_prod_protein_gene_{g}"]

    # Convert k_add_scaled → k_add
    for key in row.index:
        if "k_add_scaled" in key:
            tgt_gene = key.split("_")[7]  # target gene index
            k_on_target = converted[f"k_on_gene_{tgt_gene}"]
            converted[key.replace("k_add_scaled", "k_add")] = row[key] * k_on_target

        elif "n_gene" in key:
            converted[key] = row[key]

    return pd.Series(converted)

df_converted = df_sampled.apply(convert_params, axis=1)

# --- Expand to long format ---
rows = []
for idx, row in df_converted.iterrows():
    g1 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_1") and "to" not in k}
    g2 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_2") and "to" not in k}
    g3 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_3") and "to" not in k}
    interactions = {k: v for k, v in row.items() if "gene_" in k and "to" in k}

    rows.append({**g1, **interactions, "pair_id": idx, "gene_id": 1})
    rows.append({**g2, **interactions, "pair_id": idx, "gene_id": 2})
    rows.append({**g3, **interactions, "pair_id": idx, "gene_id": 3})

final_df = pd.DataFrame(rows).reset_index(drop=True)

# --- Save ---
output_path = (
    "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_positive_reg_pi_on_r_add_scaled.csv"
)
final_df.to_csv(output_path)
print(f"\n✅ Saved to {output_path}")


✅ Saved to /home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_positive_reg_pi_on_r_add_scaled.csv


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

# --- Define sampled parameters ---
gene_params_sampled = [
    "pi_on",            # k_on / (k_on + k_off)
    "k_on",             # directly sampled k_on
    "mrna_half_life",
    "protein_half_life",
    "k_prod_protein",
    "k_prod_mRNA"
]

interaction_params_scaled = [
    "n_gene_1_to_gene_2", "k_add_scaled_gene_1_to_gene_2",
    "n_gene_2_to_gene_1", "k_add_scaled_gene_2_to_gene_1",
    "n_gene_1_to_gene_3", "k_add_scaled_gene_1_to_gene_3",
    "n_gene_2_to_gene_3", "k_add_scaled_gene_2_to_gene_3"
]

param_names = (
    [f"{p}_gene_1" for p in gene_params_sampled] +
    [f"{p}_gene_2" for p in gene_params_sampled] +
    [f"{p}_gene_3" for p in gene_params_sampled] +
    interaction_params_scaled
)

# --- Bounds for sampled parameters ---
param_bounds = {
    # Gene-level
    "pi_on": (0.002, 0.4),                  # activation probability
    "k_on": (0.01, 3),                      # directly sampled
    "mrna_half_life": (0.6, 17),
    "protein_half_life": (7, 200),
    "k_prod_mRNA": (0.2, 60),
    "k_prod_protein": (19, 2700),
    # Interaction-level
    "n_gene_1_to_gene_2": (0.1, 5),
    "n_gene_2_to_gene_1": (0.1, 5),
    "n_gene_1_to_gene_3": (0.1, 5),
    "n_gene_2_to_gene_3": (0.1, 5),
    "k_add_scaled_gene_1_to_gene_2": (0.5, 2),
    "k_add_scaled_gene_2_to_gene_1": (0.5, 2),
    "k_add_scaled_gene_1_to_gene_3": (0.5, 2),
    "k_add_scaled_gene_2_to_gene_3": (0.5, 2),
}

bounds = (
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in interaction_params_scaled]
)

# --- Sampling configuration ---
n_valid_required = 25000
seed = 42

# Latin Hypercube Sampling (log10 for all except pi_on)
log_bounds_lower = [
    np.log10(b[0]) if "pi_on" not in param_names[i] else b[0]
    for i, b in enumerate(bounds)
]
log_bounds_upper = [
    np.log10(b[1]) if "pi_on" not in param_names[i] else b[1]
    for i, b in enumerate(bounds)
]

sampler = qmc.LatinHypercube(d=len(bounds), seed=seed)
sample = sampler.random(n=n_valid_required)

scaled_sample = np.empty_like(sample)
for i, name in enumerate(param_names):
    # Determine log bounds for every parameter (including pi_on)
    lower = np.log10(param_bounds["pi_on"][0]) if "pi_on" in name else log_bounds_lower[i]
    upper = np.log10(param_bounds["pi_on"][1]) if "pi_on" in name else log_bounds_upper[i]

    # Scale in log-space and exponentiate back
    scaled_log = qmc.scale(sample[:, [i]], [lower], [upper]).ravel()
    scaled_sample[:, i] = 10 ** scaled_log



df_sampled = pd.DataFrame(scaled_sample, columns=param_names)

# --- Convert to actual k_off and k_add ---
def convert_params(row):
    converted = {}

    for g in [1, 2, 3]:
        pi = row[f"pi_on_gene_{g}"]
        k_on = row[f"k_on_gene_{g}"]

        # Derive k_off from pi_on and k_on
        k_off = k_on * (1 - pi) / pi

        converted[f"k_on_gene_{g}"] = k_on
        converted[f"k_off_gene_{g}"] = k_off
        converted[f"mrna_half_life_gene_{g}"] = row[f"mrna_half_life_gene_{g}"]
        converted[f"protein_half_life_gene_{g}"] = row[f"protein_half_life_gene_{g}"]
        converted[f"k_prod_mRNA_gene_{g}"] = row[f"k_prod_mRNA_gene_{g}"]
        converted[f"k_prod_protein_gene_{g}"] = row[f"k_prod_protein_gene_{g}"]

    # Convert k_add_scaled → k_add
    for key in row.index:
        if "k_add_scaled" in key:
            tgt_gene = key.split("_")[7]  # target gene index
            k_on_target = converted[f"k_on_gene_{tgt_gene}"]
            converted[key.replace("k_add_scaled", "k_add")] = row[key] * k_on_target
        elif "n_gene" in key:
            converted[key] = row[key]

    return pd.Series(converted)

df_converted = df_sampled.apply(convert_params, axis=1)

# --- Expand to long format ---
rows = []
for idx, row in df_converted.iterrows():
    g1 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_1") and "to" not in k}
    g2 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_2") and "to" not in k}
    g3 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_3") and "to" not in k}
    interactions = {k: v for k, v in row.items() if "gene_" in k and "to" in k}

    rows.append({**g1, **interactions, "pair_id": idx, "gene_id": 1})
    rows.append({**g2, **interactions, "pair_id": idx, "gene_id": 2})
    rows.append({**g3, **interactions, "pair_id": idx, "gene_id": 3})

final_df = pd.DataFrame(rows).reset_index(drop=True)

# --- Save ---
output_path = (
    "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_repression_reg_pi_on_r_add_scaled.csv"
)
final_df.to_csv(output_path)
print(f"\n✅ Saved to {output_path}")


✅ Saved to /home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_repression_reg_pi_on_r_add_scaled.csv
